In [26]:
import numpy as np
import pandas as pd

train = pd.read_csv('/Users/saidattaputta/Desktop/HomeValue-AI/data/raw/train.csv')
test = pd.read_csv('/Users/saidattaputta/Desktop/HomeValue-AI/data/raw/test.csv')

In [27]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

In [28]:
def handle_missing_values(df, num_cols, cat_cols):
    cleaned_df = df.copy()
    
    for col in num_cols:
        if cleaned_df[col].isnull().sum() > 0:
            median_value = cleaned_df[col].median()
            # Direct assignment avoids the warning
            cleaned_df[col] = cleaned_df[col].fillna(median_value)
            
    for col in cat_cols:
        if cleaned_df[col].isnull().sum() > 0:
            # Direct assignment avoids the warning
            cleaned_df[col] = cleaned_df[col].fillna('None')
            
    return cleaned_df

train_clean = handle_missing_values(train, num_cols, cat_cols)
remaining_nulls_train = train_clean.isnull().sum().sum()
print(f"Total remaining missing values after cleaning: {remaining_nulls_train}")

Total remaining missing values after cleaning: 0


In [29]:
train_clean.duplicated().sum()

np.int64(0)

In [30]:
train_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1460 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          1460 non-null   object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [31]:
def handle_outliers(df, is_train=True):
    cleaned_df = df.copy()

    if is_train:
        
        cleaned_df = cleaned_df[cleaned_df['GrLivArea'] <= 4000]
        cleaned_df = cleaned_df[cleaned_df['TotalBsmtSF'] <= 6000]
        cleaned_df = cleaned_df[cleaned_df['GarageArea'] <= 1200]
        cleaned_df = cleaned_df[cleaned_df['1stFlrSF'] <= 3000]
        cleaned_df = cleaned_df[cleaned_df['SalePrice'] <= 300000]
        cleaned_df = cleaned_df[cleaned_df['LotFrontage'] <= 200]
        cleaned_df = cleaned_df[cleaned_df['LotArea'] <= 100000]
    
    return cleaned_df

print(f"Shape before comprehensive outlier removal: {train_clean.shape}")
train_clean = handle_outliers(train_clean, is_train=True)
print(f"Shape after comprehensive outlier removal: {train_clean.shape}")
print(f"percentage of data removed: {((train.shape[0] - train_clean.shape[0]) / train.shape[0]) * 100:.2f}%")

Shape before comprehensive outlier removal: (1460, 81)
Shape after comprehensive outlier removal: (1337, 81)
percentage of data removed: 8.42%


In [32]:
def handle_rare_categories(df, cat_cols, threshold=0.01, is_train=True, train_frequencies=None):
    """
    Groups infrequent categorical values into a single 'Rare' label 
    to reduce noise and prevent high-cardinality overfitting.
    """
    cleaned_df = df.copy()
    if train_frequencies is None:
        train_frequencies = {}
        
    for col in cat_cols:
        if col in cleaned_df.columns:
            if is_train:
                # 1. Calculate frequency percentage of each category in training
                freqs = cleaned_df[col].value_counts(normalize=True)
                # 2. Identify categories falling below our threshold (1%)
                rare_labels = freqs[freqs < threshold].index.tolist()
                train_frequencies[col] = rare_labels
            else:
                # If cleaning test data, use the exact rare labels found in training
                rare_labels = train_frequencies.get(col, [])
            
            # 3. Replace rare categories with 'Rare'
            cleaned_df[col] = cleaned_df[col].replace(rare_labels, 'Rare')
            
    return cleaned_df, train_frequencies

# Apply the rare category grouping
print(f"Unique values in 'Neighborhood' before: {train_clean['Neighborhood'].nunique()}")
train_clean, saved_frequencies = handle_rare_categories(train_clean, cat_cols, threshold=0.01, is_train=True)
print(f"Unique values in 'Neighborhood' after: {train_clean['Neighborhood'].nunique()}")

Unique values in 'Neighborhood' before: 25
Unique values in 'Neighborhood' after: 23


In [33]:
test_clean, _ = handle_rare_categories(test, cat_cols, threshold=0.01, is_train=False, train_frequencies=saved_frequencies)

# Now save the actually cleaned datasets
train_clean.to_csv('/Users/saidattaputta/Desktop/HomeValue-AI/data/processed/train_cleaned.csv', index=False)
test_clean.to_csv('/Users/saidattaputta/Desktop/HomeValue-AI/data/processed/test_cleaned.csv', index=False)

In [34]:
import json

# After running handle_rare_categories on your training data:
with open('/Users/saidattaputta/Desktop/HomeValue-AI/artifacts/categorical_frequencies.json', 'w') as f:
    json.dump(saved_frequencies, f)